In [2]:
import pandas as pd
import polars as pl
df_noticias1 = pd.read_csv("noticias_seu_dinheiro_paralelo_pag1_150.csv", encoding="utf-8-sig")
# df_noticias2 = pd.read_csv("noticias_seu_dinheiro_paralelo_pag151_300.csv", encoding="utf-8-sig")
# df_noticias3 = pd.read_csv("noticias_seu_dinheiro_paralelo_pag301_600.csv", encoding="utf-8-sig")
# df_noticias = pd.concat([df_noticias1, df_noticias2, df_noticias3])
df_noticias = pd.concat([df_noticias1])

df_pd = df_noticias


In [3]:
df_pd.head(5)

,url,titulo,subtitulo,data_texto,hora_texto,data_iso,autor,descricao_autor,texto,pagina_origem
0,https://www.seudinheiro.com/2026/bolsa-dolar/b...,Brasil é o emergente preferido dos estrangeiro...,"Apesar do fluxo bilionário para o Ibovespa, um...",20 de abril de 2026,13:05,2026-04-20,Monique Lima,Monique Lima é jornalista com atuação em renda...,Quem olha para o fluxo de dinheiro estrangeiro...,1
1,https://www.seudinheiro.com/2026/empresas/sequ...,Sequoia (SEQL3) reduz dívida tributária em 84%...,"Acordo com a PGFN corta passivo de R$ 631,7 mi...",20 de abril de 2026,12:42,2026-04-20,Larissa Bernardes,"Repórter no Seu Dinheiro, formada em Jornalism...",A Sequoia Logística e Transportes (SEQL3) anun...,1
2,https://www.seudinheiro.com/2026/empresas/fim-...,Fim de uma era na Braskem (BRKM5): Novonor dá ...,Venda do controle abre nova fase para a petroq...,20 de abril de 2026,12:37,2026-04-20,Camille Lima,Jornalista formada pela Universidade Municipal...,"Depois de anos de idas e vindas, a Braskem (BR...",1
3,https://www.seudinheiro.com/2026/lifestyle/eur...,Europa: novo sistema de imigração pode atrasar...,Novo sistema de entrada e saída no continente ...,20 de abril de 2026,15:30,2026-04-20,Letícia Flávia Pinheiro,Jornalista formada pela Universidade de São Pa...,"Desde o dia 10 de abril, o Entry/Exit System (...",1
4,https://www.seudinheiro.com/2026/financas-pess...,Itaú BBA aponta 14 ações com potencial de alta...,Carteira semanal do banco reúne papéis com exp...,20 de abril de 2026,15:08,2026-04-20,Money Times,NaN,A semana começa com novas indicações de ações ...,1


# Limpeza basica

In [4]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode

def limpar_texto(texto):

    if pd.isna(texto):
        return None

    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [ token for token in tokens if token.isalnum()]
    return " ".join(tokens)

df_pd['texto_limpo'] = df_pd['texto'].apply(limpar_texto)
df_pd['texto_limpo'] 



0       quem olha para o fluxo de dinheiro estrangeiro...
1       a sequoia logistica e transportes seql3 anunci...
2       depois de anos de idas e vindas a braskem brkm...
3       desde o dia 10 de abril o entry exit system ee...
4       a semana comeca com novas indicacoes de acoes ...
                              ...                        
2973    as 22h da ultima terca feira 16 quando grande ...
2974    a acao da movida movi3 mais do que triplicou d...
2975    a primeira batalha na final da copa do brasil ...
2976    o tabuleiro acionario da oncoclinicas onco3 ga...
2977    a orizon orvr3 deu um passo decisivo para muda...
Name: texto_limpo, Length: 2978, dtype: object

In [5]:
df_pd['descricao_autor_limpo'] = df_pd['descricao_autor'].apply(limpar_texto)
df_pd['descricao_autor_limpo'] 

0       monique lima e jornalista com atuacao em renda...
1       reporter no seu dinheiro formada em jornalismo...
2       jornalista formada pela universidade municipal...
3       jornalista formada pela universidade de sao pa...
4                                                    None
                              ...                        
2973    jornalista com pos graduacao em literatura art...
2974    jornalista pela universidade de sao paulo usp ...
2975    formado em jornalismo pela puc sp atua como re...
2976    jornalista formada pela universidade municipal...
2977    jornalista pela universidade de sao paulo usp ...
Name: descricao_autor_limpo, Length: 2978, dtype: object

# Removendo StopWords


In [6]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):

    if texto is None:
        return []

    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df_pd["tokens_texto_sem_stopwords"] = df_pd["texto_limpo"].apply(remover_stopwords)
df_pd["texto_sem_stopwords"] = df_pd["tokens_texto_sem_stopwords"].str.join(" ")

df_pd[["texto_limpo", "tokens_texto_sem_stopwords", "texto_sem_stopwords"]].head()



,texto_limpo,tokens_texto_sem_stopwords,texto_sem_stopwords
0,quem olha para o fluxo de dinheiro estrangeiro...,"[olha, fluxo, dinheiro, estrangeiro, entrando,...",olha fluxo dinheiro estrangeiro entrando ibove...
1,a sequoia logistica e transportes seql3 anunci...,"[sequoia, logistica, transportes, seql3, anunc...",sequoia logistica transportes seql3 anunciou n...
2,depois de anos de idas e vindas a braskem brkm...,"[anos, idas, vindas, braskem, brkm5, novo, con...",anos idas vindas braskem brkm5 novo controlado...
3,desde o dia 10 de abril o entry exit system ee...,"[desde, dia, 10, abril, entry, exit, system, e...",desde dia 10 abril entry exit system ees novo ...
4,a semana comeca com novas indicacoes de acoes ...,"[semana, comeca, novas, indicacoes, acoes, bus...",semana comeca novas indicacoes acoes busca opo...


In [7]:
df_pd["tokens_autor_sem_stopwords"] = df_pd["descricao_autor_limpo"].apply(remover_stopwords)
df_pd["autor_texto_sem_stopwords"] = df_pd["tokens_autor_sem_stopwords"].str.join(" ")

df_pd[["descricao_autor_limpo", "tokens_autor_sem_stopwords", "autor_texto_sem_stopwords"]].head()

,descricao_autor_limpo,tokens_autor_sem_stopwords,autor_texto_sem_stopwords
0,monique lima e jornalista com atuacao em renda...,"[monique, lima, jornalista, atuacao, renda, fi...",monique lima jornalista atuacao renda fixa fin...
1,reporter no seu dinheiro formada em jornalismo...,"[reporter, dinheiro, formada, jornalismo, pont...",reporter dinheiro formada jornalismo pontifici...
2,jornalista formada pela universidade municipal...,"[jornalista, formada, universidade, municipal,...",jornalista formada universidade municipal caet...
3,jornalista formada pela universidade de sao pa...,"[jornalista, formada, universidade, paulo, eca...",jornalista formada universidade paulo eca usp ...
4,None,[],


# Bag of Words

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(dtype="int16")
matriz__texto_bow = vectorizer.fit_transform(df_pd["texto_sem_stopwords"])

df__texto_bow = pd.DataFrame(
    matriz__texto_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df__texto_bow.head()

,00,000,000001,000001469,00001081,0001,00011,00013,00013527,0002,...,zudizilla,zudzilla,zuk,zukerman,zulini,zum,zumbi,zumbis,zurique,zygmunt
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
matriz__autor_bow = vectorizer.fit_transform(df_pd["autor_texto_sem_stopwords"])

df__autor_bow = pd.DataFrame(
    matriz__autor_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df__autor_bow.head()

,100,15,20,2016,2020,2021,2022,2023,2024,2025,...,vitor,vogue,volpon,voltada,webdesign,wgsn,wharton,xp,yahoo,york
0,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Retirar colunas numeros

In [10]:
colunas_com_numeros = [col for col in df__texto_bow.columns if any(char.isdigit() for char in col)]

df__texto_bow = df__texto_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df__texto_bow.head()

2892 colunas removidas


,aa,aaa,aapl,aarkhe,aaron,aatmanirbhar,ab,aba,abacate,abacaxi,...,zudizilla,zudzilla,zuk,zukerman,zulini,zum,zumbi,zumbis,zurique,zygmunt
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
colunas_com_numeros = [col for col in df__autor_bow.columns if any(char.isdigit() for char in col)]

df__autor_bow = df__autor_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df__autor_bow.head()

15 colunas removidas


,abandonado,abril,academica,acham,acioli,acoes,acumulando,acumulou,administracao,administrador,...,vitor,vogue,volpon,voltada,webdesign,wgsn,wharton,xp,yahoo,york
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Db final

In [12]:
metadados = df_pd
bow_texto_com_prefixo = df__texto_bow.add_prefix("bow_texto_").reset_index(drop=True)
bow_autor_texto_com_prefixo = df__autor_bow.add_prefix("bow_autor_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_texto_com_prefixo,bow_autor_texto_com_prefixo], axis=1)

df_final.head()

,url,titulo,subtitulo,data_texto,hora_texto,data_iso,autor,descricao_autor,texto,pagina_origem,...,bow_autor_vitor,bow_autor_vogue,bow_autor_volpon,bow_autor_voltada,bow_autor_webdesign,bow_autor_wgsn,bow_autor_wharton,bow_autor_xp,bow_autor_yahoo,bow_autor_york
0,https://www.seudinheiro.com/2026/bolsa-dolar/b...,Brasil é o emergente preferido dos estrangeiro...,"Apesar do fluxo bilionário para o Ibovespa, um...",20 de abril de 2026,13:05,2026-04-20,Monique Lima,Monique Lima é jornalista com atuação em renda...,Quem olha para o fluxo de dinheiro estrangeiro...,1,...,0,0,0,0,0,0,0,0,0,0
1,https://www.seudinheiro.com/2026/empresas/sequ...,Sequoia (SEQL3) reduz dívida tributária em 84%...,"Acordo com a PGFN corta passivo de R$ 631,7 mi...",20 de abril de 2026,12:42,2026-04-20,Larissa Bernardes,"Repórter no Seu Dinheiro, formada em Jornalism...",A Sequoia Logística e Transportes (SEQL3) anun...,1,...,0,0,0,0,0,0,0,0,0,0
2,https://www.seudinheiro.com/2026/empresas/fim-...,Fim de uma era na Braskem (BRKM5): Novonor dá ...,Venda do controle abre nova fase para a petroq...,20 de abril de 2026,12:37,2026-04-20,Camille Lima,Jornalista formada pela Universidade Municipal...,"Depois de anos de idas e vindas, a Braskem (BR...",1,...,0,0,0,0,0,0,0,0,0,0
3,https://www.seudinheiro.com/2026/lifestyle/eur...,Europa: novo sistema de imigração pode atrasar...,Novo sistema de entrada e saída no continente ...,20 de abril de 2026,15:30,2026-04-20,Letícia Flávia Pinheiro,Jornalista formada pela Universidade de São Pa...,"Desde o dia 10 de abril, o Entry/Exit System (...",1,...,0,0,0,0,0,0,0,0,0,0
4,https://www.seudinheiro.com/2026/financas-pess...,Itaú BBA aponta 14 ações com potencial de alta...,Carteira semanal do banco reúne papéis com exp...,20 de abril de 2026,15:08,2026-04-20,Money Times,NaN,A semana começa com novas indicações de ações ...,1,...,0,0,0,0,0,0,0,0,0,0


In [13]:
df_final.to_csv('df_final_v1.csv', index=False)
df_final.to_parquet('df_final_v1.parquet')
